In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB


## Lemmatization Using spaCy

The cleaned review text was lemmatized using spaCy's `en_core_web_sm` language model. Lemmatization converts words into their base forms while removing punctuation and unnecessary spacing, helping reduce vocabulary variation before feature extraction and model training.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")


def lemmatize_reviews(reviews_list):
    lemmatized = []
    for doc in nlp.pipe(reviews_list, batch_size=1000):
        tokens = [token.lemma_ for token in doc if not token.is_punct and not token.is_space]
        lemmatized.append(" ".join(tokens))
    return lemmatized

In [ ]:
from symspellpy import SymSpell, Verbosity

# Create SymSpell object
max_edit_distance = 2  # max edits allowed
prefix_length = 7      # how many letters to index
sym_spell = SymSpell(max_edit_distance, prefix_length)

# Load dictionary
dictionary_path = "../data/frequency_dictionary_en_82_765.txt"
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)



In [ ]:
def correct_sentence(sentence):
    corrected_words = []
    for word in sentence.split():
        suggestions = sym_spell.lookup(word, Verbosity.TOP, max_edit_distance)
        if suggestions:
            corrected_words.append(suggestions[0].term)  # take most probable correction
        else:
            corrected_words.append(word)  # keep original if unknown
    return " ".join(corrected_words)


In [ ]:
sentiment_classifier_data=pd.read_csv("../data/corrected_reviews_symspell01.csv", encoding='latin1')

In [ ]:
sentiment_classifier_data.head()

In [ ]:
sentiment_classifier_data.tail()

In [ ]:
sentiment_classifier_data["sentiment"].value_counts()

In [ ]:
sentiment_classifier_data=sentiment_classifier_data.drop_duplicates()

In [ ]:
sentiment_classifier_data["sentiment"].value_counts()

In [ ]:
sentiment_classifier_data["review"]=sentiment_classifier_data["review"].str.strip()
sentiment_classifier_data["review"]=sentiment_classifier_data["review"].str.replace(r'\s+', ' ', regex=True)
sentiment_classifier_data["review"]=sentiment_classifier_data["review"].str.lower()
sentiment_classifier_data["review"]=sentiment_classifier_data["review"].str.replace(r'[^a-zA-Z0-9.,!? ]', '', regex=True)

In [ ]:
sentiment_classifier_data.head()

In [ ]:
# Save to CSV
sentiment_classifier_data.to_csv("corrected_reviews_symspell01.csv", index=False)


In [ ]:

X=sentiment_classifier_data['review']
Y=sentiment_classifier_data['sentiment']

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.1,stratify=Y,random_state=1)

In [ ]:
X_train_clean = lemmatize_reviews(X_train)
X_test_clean  = lemmatize_reviews(X_test)

In [ ]:
# Save train
train_df = pd.DataFrame({
    "review": X_train_clean,
    "label": Y_train
})
train_df.to_csv("train_lemmatized.csv", index=False)

# Save test
test_df = pd.DataFrame({
    "review": X_test_clean,
    "label": Y_test
})
test_df.to_csv("test_lemmatized.csv", index=False)

In [ ]:
vectorizer=TfidfVectorizer(ngram_range=(1,2), stop_words='english',min_df=10,max_df=0.9)
X_train_vec=vectorizer.fit_transform(X_train_clean)
X_test_vec=vectorizer.transform(X_test_clean)

In [ ]:
print(X.shape,X_train.shape,X_test.shape)

In [ ]:
model=MultinomialNB()



In [ ]:
model.fit(X_train_vec,Y_train)

In [ ]:
X_train_prediction=model.predict(X_train_vec)
training_data_accuracy=accuracy_score(X_train_prediction,Y_train)

print("Accuracy on training data:",training_data_accuracy)

In [ ]:
X_test_prediction=model.predict(X_test_vec)
test_data_accuracy=accuracy_score(X_test_prediction,Y_test)

print("Accuracy on testing data:",test_data_accuracy)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, X_test_prediction))

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(Y_test, X_test_prediction)
print(cm)
